# 05 - AI analyst: grounded summaries and Q&A
_Synthetic data._

The model never invents numbers: it writes from a fact sheet and every number is machine-checked against the facts it cites. This notebook runs offline (no API key needed).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from python import visualize as viz   # shared chart style (palette, fonts, grid)
%matplotlib inline
from python.config import get_settings
S = get_settings(base_dir=ROOT)
con = duckdb.connect(str(S.warehouse_path), read_only=True)
q = lambda sql: con.execute(sql).df()
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
print("Warehouse:", S.warehouse_path.name, "| as of", S.end_date, "| SYNTHETIC DATA")

Warehouse: fintech.duckdb | as of 2026-06-30 | SYNTHETIC DATA


In [2]:
from ai_analyst.fact_sheet import build_fact_sheet, facts_for_prompt
sheet = build_fact_sheet(S)
print(facts_for_prompt(sheet)[:2500])

[F01] (kpi_monthly) New customers (Jun 2026): 4,256; prior month 4,158 (+2.4%); same month last year 3,182 (+33.8%); assessment: stable.
[F02] (kpi_monthly) KYC completion (7-day) (Jun 2026): 62.3%; prior month 63.2% (-0.9 pp); same month last year 56.3% (+6.0 pp); target >= 62.0% -> on track; assessment: stable.
[F03] (kpi_monthly) Activation rate (first loan in 30 days) (May 2026): 17.0%; prior month 18.0% (-1.0 pp); same month last year 15.7% (+1.3 pp); target >= 15.0% -> on track; assessment: stable. (Latest available period: Needs 30 days of observation, so the current month is reported next month..)
[F04] (kpi_monthly) Applications submitted (Jun 2026): 2,271; prior month 2,271 (+0.0%); same month last year 1,475 (+54.0%); assessment: stable.
[F05] (kpi_monthly) Approval rate (Jun 2026): 62.2%; prior month 64.6% (-2.4 pp); same month last year 60.5% (+1.7 pp); target >= 50.0% -> on track; assessment: down, notable.
[F06] (kpi_monthly) Offer acceptance (disbursal rate) (Jun 2026):

## The grounding checker catches an invented number

In [3]:
from ai_analyst.grounding import validate_items
f = next(x for x in sheet['facts'] if x['data'].get('kpi') == 'approval_rate')
good = {'section':'demo','text': 'Approval rate: ' + f['data']['value_fmt'], 'fact_ids':[f['id']]}
bad  = {'section':'demo','text': 'Approval rate jumped to 71.3%', 'fact_ids':[f['id']]}
validate_items([good, bad], sheet['facts'])

{'grounded': False,
 'items_checked': 2,
 'numbers_checked': 2,
 'problems': [{'section': 'demo',
   'text': 'Approval rate jumped to 71.3%',
   'issue': 'numbers not found in cited facts',
   'detail': ['71.3%'],
   'cited': ['F05']}]}

## The SQL guard blocks unsafe model-written SQL

In [4]:
from ai_analyst.sql_guard import validate_sql, SQLGuardError
for s in ['SELECT COUNT(*) FROM marts.fct_loan_performance', 'DROP TABLE core.loans', "SELECT * FROM read_csv('secrets.csv')"]:
    try: print('OK     ', validate_sql(s, ['marts.fct_loan_performance']).splitlines()[1])
    except SQLGuardError as e: print('BLOCKED', s, '->', e)

OK      SELECT COUNT(*) FROM marts.fct_loan_performance
BLOCKED DROP TABLE core.loans -> only SELECT / WITH queries are allowed
BLOCKED SELECT * FROM read_csv('secrets.csv') -> file / external functions are not allowed: read_csv


## Weekly executive summary (deterministic mode; set ANTHROPIC_API_KEY to let Claude write it)

In [5]:
from ai_analyst.summarizer import generate_summary
r = generate_summary(S, offline=True)
print(r['mode'], r['validation']['grounded'], r['validation']['numbers_checked'], 'numbers checked')
from IPython.display import Markdown
Markdown(Path(r['path']).read_text().split('## Sources')[0])

deterministic True 116 numbers checked


# Weekly Business Summary - week of 22 Jun 2026

_Vittora Credit (fictional) - synthetic data - generated 2026-10-03 03:19 - engine: deterministic template (no LLM call) - numeric verification: **PASSED** (116 numbers in 37 items checked against the cited facts)_

> **Jun 2026: Gross revenue â‚¹59.30 L (+1.6% vs prior month); Loans disbursed 1,293 (-2.1% vs prior month); Collection efficiency remains below target at 93.9%.** [F13][F07][F20]

## Highlights

- Week of 22 Jun: Loans disbursed 317 (+7.1% week on week); Disbursed amount â‚¹1.41 Cr (+4.7% week on week); EMI collections â‚¹1.38 Cr (+16.6% week on week); Payment failure rate 11.2% (+0.8 pp week on week). [F33][F34][F35][F36]
- Disbursed amount â‚¹5.86 Cr (+1.3% vs prior month); Net revenue â‚¹33.96 L (-17.2% vs prior month). [F08][F15]
- KYC completion (7-day) 62.3% (-0.9 pp vs prior month); Approval rate 62.2% (-2.4 pp vs prior month). [F02][F05]
- PAR30 3.6% (-0.2 pp vs prior month); NPA (90+ DPD) 2.0% (-0.5 pp vs prior month). [F17][F18]
- Monthly active users 14,918 (+1.4% vs prior month); Repeat-loan share 34.9% (-3.7 pp vs prior month). [F22][F11]
- Onboarding redesign experiment: 7-day KYC completion 61.78% treatment vs 58.51% control, difference +3.26 pp (95% CI +0.54 to +5.99 pp), relative uplift +5.6%, p-value 0.0191; P(treatment better) 99.0%. [F50]

## KPI movements

- Approval rate: 62.2% (-2.4 pp vs prior month) - down. [F05]
- Repeat-loan share: 34.9% (-3.7 pp vs prior month) - worsened. [F11]
- Credit losses (write-offs): â‚¹25.34 L (+46.1% vs prior month) - worsened. [F14]
- Net revenue: â‚¹33.96 L (-17.2% vs prior month) - worsened. [F15]
- Support tickets: 361 (+7.8% vs prior month) - worsened. [F23]
- Tickets per 1k MAU: 24.20 (+6.2% vs prior month) - worsened. [F24]
- Marketing spend: â‚¹11.62 L (+7.4% vs prior month) - up. [F26]

## Anomalies

- No anomaly episodes in the last 8 weeks; the monitor remains active. [F41]
- Past incident for context: EMI payment failure rate spiked to 47.1% on 15 Aug 2025 (expected 8.9%, robust z 15.6); main driver: failure reason (excess failed debits) = gateway_timeout (96% of the deviation). Window 2025-08-14 to 2025-08-15, 2 day(s) flagged, severity high (no known business event - investigate). [F46]
- Past incident for context: Signups spiked to 222 on 15 Sep 2025 (expected 113, robust z 6.8); main driver: acquisition channel = affiliate (87% of the deviation). Window 2025-09-08 to 2025-09-21, 14 day(s) flagged, severity high (no known business event - investigate). [F47]
- Past incident for context: Daily active users spiked to 920 on 16 Sep 2025 (expected 692, robust z 5.8); main driver: platform = android (94% of the deviation). Window 2025-09-13 to 2025-09-20, 6 day(s) flagged, severity high (no known business event - investigate). [F48]

## Experiments

- EXP-ONB-2026-01 guardrails: approval rate +3.13 pp (pass), FPD30 -1.48 pp (pass), KYC support tickets +0.16 pp (inconclusive). Decision: SHIP with post-launch monitoring of kyc_support_ticket_rate. [F52]
- EXP-RPY-2025-11 (Autopay nudge at disbursal): autopay 64.28% treatment vs 46.62% control, difference +17.65 pp (95% CI +12.76 to +22.54 pp), p-value 0.0000. Decision: SHIP. [F54]
- EXP-PRC-2025-06 (Upfront fee transparency on the offer screen): accepted 90.98% treatment vs 88.78% control, difference +2.20 pp (95% CI -1.93 to +6.32 pp), p-value 0.2974. Decision: INCONCLUSIVE - underpowered: needed 1,530 users per arm for a 3.0 pp MDE, had 410. [F55]

## Risks

- Collection efficiency is off target: 93.9% (>= 95.0% -> off track). [F20]
- Payment failure rate is off target: 10.3% (<= 8.0% -> off track). [F21]
- CSAT is off target: 3.46 (>= 4.00 -> off track). [F25]
- Festive-season credit dilution: First-loan FPD30 for loans disbursed in the Oct-Nov 2025 festive season vs Jun-Sep 2025: affiliate 4.49% vs 1.71%, paid social 5.42% vs 3.67%, partnerships 2.40% vs 3.25%. [F59]
- Acquisition fraud exposure: Campaign CMP-202509-AFFNB (Affiliate Network B, Sep 2025): spend â‚¹5.56 L, 1,258 signups, 4 funded customers, cost per funded customer â‚¹1.39 L; flagged 'inefficient_spend'. [F61]

## Recommendations

| # | Recommendation | Owner | Evidence |
|---|---|---|---|
| 1 | Add pre-due-date reminders (three days and one day before the EMI date) and same-day retries for failed auto-debits. | Collections & Repayments | [F20] |
| 2 | Make autopay mandate set-up the default step at disbursal and schedule debit retries around salary-credit dates; autopay borrowers repay on time more often. | Collections & Repayments | [F21][F60] |
| 3 | Attack the top ticket drivers (KYC issues, payment failures) with in-app self-serve fixes and a first-response SLA; CSAT is below target. | Customer Support | [F25] |
| 4 | Shift paid budget from affiliate and paid social towards referral and employer partnerships, which return the most value per rupee of acquisition cost. | Marketing | [F28][F58] |
| 5 | Add device and velocity rules for affiliate traffic and pay affiliates on funded, performing loans instead of signups. | Risk & Marketing | [F61] |

## Data caveats

- Data quality: DQ score 92.0 on the raw extract and 99.1 on the curated layer; critical failures 22 -> 0; 9 warnings remain (documented open issues); 27 of 27 planted defect types detected. [F64]
- Open data-quality issue (warning): marketing_campaigns.budget_inr,spend_inr cross_field:spend_within_budget - 1 rows (spend_inr <= 1.5 * budget_inr). [F65]
- Open data-quality issue (warning): customers.city not_null - 974 rows (nulls in a non-nullable column). [F66]
- Open data-quality issue (warning): customers.age not_null - 65 rows (nulls in a non-nullable column). [F67]
- Cohort-based KPIs report with a lag: Activation rate (first loan in 30 days) refers to May 2026, First-payment default (FPD30) refers to Apr 2026. [F03][F19]
- All figures are synthetic and generated for demonstration.



In [6]:
from ai_analyst.qa import ask
ask('What was the PAR30 and NPA last month?', S, offline=True)['answer']

{'answer': 'PAR30 for Jun 2026 was 3.6% (prior month 3.8%, -0.2 pp). Target <= 6.0%: on track. Definition: Outstanding principal on loans 30-179 DPD / total outstanding at month end. NPA (90+ DPD) for Jun 2026 was 2.0% (prior month 2.5%, -0.5 pp). Target <= 3.5%: on track. Definition: Outstanding principal on loans 90-179 DPD / total outstanding at month end (RBI NPA definition).',
 'key_numbers': [{'value_text': '3.6%', 'query_id': 'kpi_snapshot'},
  {'value_text': '2.0%', 'query_id': 'kpi_snapshot'}],
 'caveats': ['Answered from the governed KPI snapshot (offline mode); synthetic data.']}